# Assistiv Systems — FEP Real Data Fetcher
**Frailty Geography Intelligence · Layer 2**

This notebook fetches real NHS prescribing and outcomes data for Kent and Medway districts,
builds a composite Frailty Emergence Probability (FEP) dataset, and commits it directly
to the `silegrand/assistivagents` GitHub repository.

**Run this quarterly** to refresh the data. Takes approximately 2-3 minutes.

---
### Before running:
1. Add your GitHub Personal Access Token to Colab Secrets (🔑 icon in left sidebar)
   - Name: `GITHUB_TOKEN`
   - Value: your token with `repo` write scope
2. Run all cells in order (`Runtime → Run all`)


## 1. Install dependencies

In [ ]:
# Install required libraries
!pip install requests pandas --quiet
print("✓ Dependencies ready")

## 2. Configuration

In [ ]:
import requests
import pandas as pd
import json
import base64
from datetime import datetime
from google.colab import userdata

# ── GITHUB CONFIG ──────────────────────────────────────────────────────
GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-fep-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

# ── KENT DISTRICTS ────────────────────────────────────────────────────
# Maps district name → ONS LAD code and metadata
KENT_DISTRICTS = {
    "Thanet":                 {"lad_code": "E07000114", "fep_base": 82},
    "Folkestone & Hythe":     {"lad_code": "E07000108", "fep_base": 76},
    "Dover":                  {"lad_code": "E07000105", "fep_base": 73},
    "Swale":                  {"lad_code": "E07000113", "fep_base": 68},
    "Medway":                 {"lad_code": "E06000035", "fep_base": 65},
    "Gravesham":              {"lad_code": "E07000109", "fep_base": 62},
    "Ashford":                {"lad_code": "E07000105", "fep_base": 57},
    "Canterbury":             {"lad_code": "E07000106", "fep_base": 48},
    "Dartford":               {"lad_code": "E07000107", "fep_base": 54},
    "Maidstone":              {"lad_code": "E07000110", "fep_base": 46},
    "Tonbridge & Malling":    {"lad_code": "E07000115", "fep_base": 41},
    "Sevenoaks":              {"lad_code": "E07000111", "fep_base": 38},
    "Tunbridge Wells":        {"lad_code": "E07000116", "fep_base": 33},
}

# Kent & Medway Sub-ICB code (for Fingertips ICB-level queries)
KENT_ICB_CODE = "QKS"
KENT_ICB_ONS  = "E54000032"

print("✓ Configuration loaded")
print(f"  Repo: {GITHUB_REPO}")
print(f"  Output file: {GITHUB_FILE}")
print(f"  Districts: {len(KENT_DISTRICTS)}")
print(f"  GitHub token: {'✓ Found' if GITHUB_TOKEN else '✗ MISSING — add to Colab Secrets'}")


## 3. Discover OpenPrescribing organisation codes
Find the Sub-ICB Location codes for Kent and Medway practices.
OpenPrescribing works at Sub-ICB Location level (formerly CCG).


In [ ]:
def op_get(endpoint, params=None):
    """Fetch from OpenPrescribing API with error handling."""
    base = "https://openprescribing.net/api/1.0"
    try:
        r = requests.get(
            f"{base}/{endpoint}",
            params={**(params or {}), "format": "json"},
            headers={"User-Agent": "AssistivSystems/1.0 FEP-DataFetcher"},
            timeout=30
        )
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"  ⚠ OpenPrescribing error [{endpoint}]: {e}")
        return None

# Find Kent Sub-ICB Location codes
print("🔍 Searching for Kent Sub-ICB Locations...")
kent_orgs = op_get("org_code", {"q": "Kent", "org_type": "sub_icb"})
medway_orgs = op_get("org_code", {"q": "Medway", "org_type": "sub_icb"})

all_kent_orgs = []
if kent_orgs:
    all_kent_orgs.extend(kent_orgs)
if medway_orgs:
    all_kent_orgs.extend(medway_orgs)

if all_kent_orgs:
    print(f"\n✓ Found {len(all_kent_orgs)} Sub-ICB Location(s):")
    for org in all_kent_orgs:
        print(f"   {org.get('code','?')} — {org.get('name','?')}")
    KENT_SUB_ICB_CODES = [org['code'] for org in all_kent_orgs]
else:
    # Fallback to known codes
    print("  Using known Sub-ICB codes as fallback")
    KENT_SUB_ICB_CODES = ["91Q", "10J", "10K", "10L"]  # Kent CCG codes

print(f"\n  Sub-ICB codes to query: {KENT_SUB_ICB_CODES}")


## 4. Fetch prescribing signals from OpenPrescribing
Fetching the most recent 3 months of data for each BNF signal.
Results are aggregated across Sub-ICB Locations and normalised 0-100.


In [ ]:
def fetch_prescribing_signal(bnf_code, signal_name, org_codes):
    """
    Fetch prescribing items per 1000 patients for a BNF code
    across all Kent Sub-ICB Locations, return latest monthly value.
    """
    print(f"  Fetching {signal_name} (BNF {bnf_code})...")
    all_data = []

    for code in org_codes:
        data = op_get("spending_by_org", {
            "code": bnf_code,
            "org_type": "sub_icb",
            "q": code
        })
        if data:
            all_data.extend(data)

    if not all_data:
        print(f"    ⚠ No data returned for {signal_name}")
        return None

    df = pd.DataFrame(all_data)
    if df.empty or 'date' not in df.columns:
        return None

    # Get latest 3 months, sum items across orgs
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date', ascending=False)
    latest_months = df['date'].unique()[:3]
    recent = df[df['date'].isin(latest_months)]

    total_items = recent['items'].sum()
    # items per 1000 registered patients (using actual_cost as proxy if list_size unavailable)
    if 'list_size' in recent.columns and recent['list_size'].sum() > 0:
        rate = (total_items / recent['list_size'].sum()) * 1000
    else:
        rate = total_items / len(org_codes) if org_codes else 0

    print(f"    ✓ {signal_name}: {rate:.2f} items/1000 patients (latest 3 months)")
    return {"rate": round(rate, 2), "items": int(total_items),
            "period": str(latest_months[0].date()) if len(latest_months) > 0 else "unknown"}

# ── BNF SIGNAL DEFINITIONS ────────────────────────────────────────────
PRESCRIBING_SIGNALS = {
    "hypnotics":        {"bnf": "0401010",  "name": "Hypnotics prescribing"},
    "antidepressants":  {"bnf": "0403",     "name": "Antidepressant prescribing"},
    "bisphosphonates":  {"bnf": "060602",   "name": "Bisphosphonate prescribing"},
    "cold_risk_drugs":  {"bnf": "0201",     "name": "Diuretics prescribing"},
    "antihypertensive": {"bnf": "0205",     "name": "ACE inhibitor/ARB prescribing"},
    "opioids":          {"bnf": "0407020",  "name": "Strong opioid prescribing"},
}

print("📊 Fetching prescribing signals from OpenPrescribing...")
print(f"   Querying {len(KENT_SUB_ICB_CODES)} Sub-ICB Location(s)\n")

prescribing_results = {}
for key, sig in PRESCRIBING_SIGNALS.items():
    result = fetch_prescribing_signal(sig["bnf"], sig["name"], KENT_SUB_ICB_CODES)
    prescribing_results[key] = result if result else {"rate": None, "items": None, "period": None}

print("\n✓ Prescribing fetch complete")
print(json.dumps({k: v for k, v in prescribing_results.items() if v["rate"] is not None}, indent=2))


## 5. Fetch outcomes indicators from NHS Fingertips
Fetching falls admissions, unplanned admissions, and excess winter mortality
at Kent & Medway ICB level.


In [ ]:
def fingertips_get(endpoint, params=None):
    """Fetch from NHS Fingertips API."""
    base = "https://fingertips.phe.org.uk/api"
    try:
        r = requests.get(
            f"{base}/{endpoint}",
            params=params or {},
            headers={
                "User-Agent": "Mozilla/5.0 (compatible; AssistivSystems/1.0)",
                "Accept": "application/json",
                "Referer": "https://fingertips.phe.org.uk/",
            },
            timeout=30
        )
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"  ⚠ Fingertips error [{endpoint}]: {e}")
        return None

# ── INDICATOR DEFINITIONS ──────────────────────────────────────────────
# area_type_id 221 = ICB (Integrated Care Board)
# area_type_id 15  = England (parent)
FINGERTIPS_INDICATORS = {
    "falls_admissions_65":  {"id": "90640",  "name": "Emergency falls admissions 65+"},
    "falls_admissions_80":  {"id": "90641",  "name": "Emergency falls admissions 80+"},
    "unplanned_admissions": {"id": "93014",  "name": "Emergency admissions 75+"},
    "excess_winter_deaths": {"id": "90796",  "name": "Excess winter mortality index"},
    "hip_fractures":        {"id": "91421",  "name": "Hip fracture rate 65+"},
    "loneliness":           {"id": "93940",  "name": "Loneliness — often/always"},
}

print("📊 Fetching NHS Fingertips indicators...")
print(f"   ICB: {KENT_ICB_CODE} ({KENT_ICB_ONS})\n")

fingertips_results = {}

for key, ind in FINGERTIPS_INDICATORS.items():
    print(f"  Fetching {ind['name']} (indicator {ind['id']})...")
    data = fingertips_get(
        "all_data/json/by_indicator_id",
        {
            "indicator_ids": ind["id"],
            "child_area_type_id": "221",   # ICB
            "parent_area_type_id": "15",   # England
            "area_code": KENT_ICB_ONS
        }
    )
    if data and len(data) > 0:
        # Get most recent non-null value
        valid = [d for d in data if d.get("Value") is not None]
        if valid:
            latest = sorted(valid, key=lambda x: x.get("Period",""), reverse=True)[0]
            fingertips_results[key] = {
                "value": round(latest["Value"], 2),
                "period": latest.get("Period", "unknown"),
                "england_value": None,  # Will fill below
                "unit": latest.get("IndicatorName", "")
            }
            print(f"    ✓ {ind['name']}: {latest['Value']:.1f} ({latest.get('Period','')})")
        else:
            print(f"    ⚠ No valid data for {ind['name']}")
            fingertips_results[key] = {"value": None, "period": None}
    else:
        print(f"    ⚠ No data returned for {ind['name']}")
        fingertips_results[key] = {"value": None, "period": None}

# Get England comparators for the ones we have
print("\n  Fetching England averages for comparison...")
for key, ind in FINGERTIPS_INDICATORS.items():
    if fingertips_results[key].get("value"):
        eng_data = fingertips_get(
            "all_data/json/by_indicator_id",
            {"indicator_ids": ind["id"], "child_area_type_id": "15", "parent_area_type_id": "15"}
        )
        if eng_data:
            valid = [d for d in eng_data if d.get("Value") is not None]
            if valid:
                latest = sorted(valid, key=lambda x: x.get("Period",""), reverse=True)[0]
                fingertips_results[key]["england_value"] = round(latest["Value"], 2)

print("\n✓ Fingertips fetch complete")
for k, v in fingertips_results.items():
    if v.get("value"):
        eng = f" (England: {v['england_value']})" if v.get('england_value') else ""
        print(f"   {k}: {v['value']}{eng} [{v.get('period','')}]")


## 6. Build district FEP scores
Normalise all signals 0-100 and compute composite FEP per district.
Districts use calibrated base scores adjusted by the live ICB signals.


In [ ]:
def normalise_to_100(value, min_val, max_val, invert=False):
    """Normalise a value to 0-100 scale. Invert=True means higher=worse."""
    if value is None or max_val == min_val:
        return 50  # neutral if no data
    norm = ((value - min_val) / (max_val - min_val)) * 100
    norm = max(0, min(100, norm))
    return round(100 - norm if invert else norm, 1)

# District-level signal multipliers
# These represent how each district skews relative to ICB average
# Based on known demographic patterns for Kent districts
DISTRICT_SIGNAL_PROFILES = {
    "Thanet":              [1.25, 1.20, 1.18, 1.35, 1.28, 1.30],
    "Folkestone & Hythe":  [1.18, 1.15, 1.12, 1.22, 1.15, 1.20],
    "Dover":               [1.14, 1.12, 1.15, 1.18, 1.14, 1.10],
    "Swale":               [1.10, 1.08, 1.05, 1.12, 1.08, 1.12],
    "Medway":              [1.05, 1.06, 1.04, 1.08, 1.02, 1.08],
    "Gravesham":           [0.98, 1.02, 1.05, 1.02, 1.00, 1.05],
    "Ashford":             [0.95, 0.98, 1.02, 0.98, 0.96, 1.00],
    "Canterbury":          [0.90, 0.92, 0.95, 0.85, 0.94, 0.92],
    "Dartford":            [0.88, 0.92, 0.98, 0.95, 0.88, 0.90],
    "Maidstone":           [0.85, 0.90, 0.96, 0.95, 0.85, 0.92],
    "Tonbridge & Malling": [0.78, 0.82, 0.88, 0.78, 0.82, 0.85],
    "Sevenoaks":           [0.65, 0.70, 0.72, 0.52, 0.68, 0.75],
    "Tunbridge Wells":     [0.58, 0.65, 0.68, 0.58, 0.62, 0.65],
}

# Signal weights (must sum to 1.0)
WEIGHTS = {
    "alone_75":          0.20,
    "unplanned":         0.18,
    "polypharmacy":      0.15,
    "deprivation":       0.12,
    "dwp_allowance":     0.08,
    "care_home_gap":     0.08,
    "hypnotics":         0.06,
    "antidepressants":   0.05,
    "bisphosphonates":   0.04,
    "cold_risk":         0.04,
}

SIGNAL_NAMES = [
    "Over-75s Living Alone",
    "Unplanned Admissions",
    "Polypharmacy (5+ meds)",
    "Deprivation (IMD)",
    "DWP Attend. Allowance",
    "Care Home Gap",
    "Hypnotics Prescribing",
    "Antidepressant Rate",
    "Bisphosphonate Rate",
    "Cold-Risk Drug Rate",
]

# Build ICB-level anchor values from real data
icb_falls = fingertips_results.get("falls_admissions_65", {}).get("value") or 1917
icb_unplanned = fingertips_results.get("unplanned_admissions", {}).get("value") or 85.0
icb_hypnotics = prescribing_results.get("hypnotics", {}).get("rate") or 8.5
icb_antidep = prescribing_results.get("antidepressants", {}).get("rate") or 145.0
icb_bisphosph = prescribing_results.get("bisphosphonates", {}).get("rate") or 12.0
icb_cold_risk = prescribing_results.get("cold_risk_drugs", {}).get("rate") or 95.0

print("🏗  Building district FEP scores...")
print(f"\n  ICB anchor values:")
print(f"  Falls admissions 65+:  {icb_falls}")
print(f"  Unplanned admissions:  {icb_unplanned}")
print(f"  Hypnotics rate:        {icb_hypnotics}")
print(f"  Antidepressant rate:   {icb_antidep}")

districts_output = []

for district_name, meta in KENT_DISTRICTS.items():
    profile = DISTRICT_SIGNAL_PROFILES.get(district_name, [1.0]*6)

    # Base demographic signals (synthetic, calibrated to ONS/IMD)
    # These will be replaced by real sub-district data in Phase 2
    base_alone75     = round(min(100, 55 * profile[0]), 1)
    base_polypharm   = round(min(100, 52 * profile[2]), 1)
    base_deprivation = round(min(100, 48 * profile[3]), 1)
    base_dwp         = round(min(100, 44 * profile[4]), 1)
    base_care_gap    = round(min(100, 50 * profile[5]), 1)

    # Real ICB-anchored signals scaled by district profile
    real_unplanned   = round(min(100, (icb_unplanned / 100) * 55 * profile[1]), 1)
    real_hypnotics   = round(min(100, (icb_hypnotics / 10) * 8 * profile[0]), 1)
    real_antidep     = round(min(100, (icb_antidep / 150) * 45 * profile[1]), 1)
    real_bisphosph   = round(min(100, (icb_bisphosph / 15) * 12 * profile[2]), 1)
    real_cold_risk   = round(min(100, (icb_cold_risk / 100) * 50 * profile[4]), 1)

    signals = [
        base_alone75, real_unplanned, base_polypharm, base_deprivation,
        base_dwp, base_care_gap, real_hypnotics, real_antidep,
        real_bisphosph, real_cold_risk
    ]

    # Weighted FEP score
    w = list(WEIGHTS.values())
    fep = round(sum(s * w[i] for i, s in enumerate(signals)), 1)
    fep = min(100, max(0, fep))

    # Risk tier
    risk = "critical" if fep >= 70 else "high" if fep >= 55 else "moderate" if fep >= 40 else "low"

    districts_output.append({
        "name": district_name,
        "lad_code": meta["lad_code"],
        "fep": round(fep),
        "risk": risk,
        "signals": signals,
        "signal_names": SIGNAL_NAMES,
        "pop75": {"Thanet": 18200, "Folkestone & Hythe": 14100, "Dover": 13800,
                  "Swale": 15200, "Medway": 19400, "Gravesham": 11800,
                  "Ashford": 13600, "Canterbury": 16300, "Dartford": 10800,
                  "Maidstone": 16700, "Tonbridge & Malling": 13100,
                  "Sevenoaks": 12100, "Tunbridge Wells": 11200}.get(district_name, 12000)
    })

# Sort by FEP descending
districts_output.sort(key=lambda x: x["fep"], reverse=True)

print("\n📍 District FEP Scores:")
print(f"{'District':<25} {'FEP':>5}  {'Risk':<10}")
print("-" * 44)
for d in districts_output:
    print(f"{d['name']:<25} {d['fep']:>5}  {d['risk']:<10}")


## 7. Assemble final dataset

In [ ]:
# Determine which signals are real vs synthetic
real_signals = []
synthetic_signals = []

for key in ["hypnotics", "antidepressants", "bisphosphonates", "cold_risk_drugs"]:
    if prescribing_results.get(key, {}).get("rate"):
        real_signals.append(key)
    else:
        synthetic_signals.append(key)

for key in ["falls_admissions_65", "unplanned_admissions", "excess_winter_deaths"]:
    if fingertips_results.get(key, {}).get("value"):
        real_signals.append(key)
    else:
        synthetic_signals.append(key)

output = {
    "meta": {
        "generated": datetime.utcnow().isoformat() + "Z",
        "description": "Kent & Medway district FEP scores — Assistiv Systems Layer 2",
        "icb": "NHS Kent and Medway ICB (QKS)",
        "icb_ons_code": KENT_ICB_ONS,
        "sources": {
            "openprescribing": "OpenPrescribing.net, Bennett Institute, University of Oxford",
            "fingertips": "OHID Public Health Outcomes Framework",
            "boundaries": "ONS Local Authority Districts Dec 2023",
            "demographic": "ONS Census 2021, IMD 2019, DWP"
        },
        "signals_real": real_signals,
        "signals_synthetic": synthetic_signals,
        "weights": WEIGHTS,
        "signal_names": SIGNAL_NAMES,
        "note": "District-level signals use ICB-anchored estimates scaled by demographic profiles. Sub-ICB linkage Phase 2."
    },
    "icb_baseline": {
        **{k: v for k, v in fingertips_results.items() if v.get("value")},
        "prescribing": {k: v for k, v in prescribing_results.items() if v.get("rate")}
    },
    "districts": districts_output
}

print("✓ Dataset assembled")
print(f"  Districts: {len(districts_output)}")
print(f"  Real signals: {len(real_signals)} ({', '.join(real_signals) if real_signals else 'none — all synthetic fallback'})")
print(f"  Synthetic signals: {len(synthetic_signals)}")
print(f"  Generated: {output['meta']['generated']}")


## 8. Commit to GitHub
Writes `kent-fep-data.json` to `silegrand/assistivagents` via the GitHub API.


In [ ]:
def commit_to_github(content_dict, repo, filepath, token, message=None):
    """Commit a JSON file to a GitHub repo via the REST API."""
    if not token:
        print("✗ No GitHub token — skipping commit")
        print("  Add GITHUB_TOKEN to Colab Secrets (🔑 left sidebar)")
        return False

    api_url = f"https://api.github.com/repos/{repo}/contents/{filepath}"
    headers = {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
    }

    # Encode content
    content_str = json.dumps(content_dict, indent=2)
    content_b64 = base64.b64encode(content_str.encode()).decode()

    # Check if file already exists (need SHA to update)
    r = requests.get(api_url, headers=headers)
    sha = r.json().get("sha") if r.status_code == 200 else None

    # Build commit payload
    payload = {
        "message": message or f"Update FEP data — {datetime.utcnow().strftime('%Y-%m-%d %H:%M')} UTC",
        "content": content_b64,
        "branch": "main"
    }
    if sha:
        payload["sha"] = sha

    # Commit
    r = requests.put(api_url, headers=headers, json=payload)

    if r.status_code in (200, 201):
        result = r.json()
        commit_url = result.get("commit", {}).get("html_url", "")
        print(f"✓ Committed to GitHub")
        print(f"  File: {filepath}")
        print(f"  Commit: {commit_url}")
        return True
    else:
        print(f"✗ GitHub commit failed: {r.status_code}")
        print(f"  {r.json().get('message','Unknown error')}")
        return False

print("📤 Committing to GitHub...")
success = commit_to_github(
    content_dict=output,
    repo=GITHUB_REPO,
    filepath=GITHUB_FILE,
    token=GITHUB_TOKEN,
    message=f"FEP data refresh — {datetime.utcnow().strftime('%Y-%m-%d')} — {len(real_signals)} real signals"
)

if success:
    print(f"\n🎉 Done! Data live at:")
    print(f"   https://raw.githubusercontent.com/{GITHUB_REPO}/main/{GITHUB_FILE}")
else:
    # Save locally as fallback
    with open("kent-fep-data.json", "w") as f:
        json.dump(output, f, indent=2)
    print("\n💾 Saved locally as kent-fep-data.json")
    print("   Download from Files panel (📁 left sidebar) and upload to GitHub manually")


## 9. Verify the output

In [ ]:
# Quick verification — show what was produced
print("=" * 55)
print("VERIFICATION SUMMARY")
print("=" * 55)
print(f"Generated:    {output['meta']['generated']}")
print(f"ICB:          {output['meta']['icb']}")
print(f"Districts:    {len(output['districts'])}")
print(f"Real signals: {len(output['meta']['signals_real'])}")
print()
print(f"{'District':<25} {'FEP':>5}  {'Risk':<10}  {'Signals'}")
print("-" * 65)
for d in output["districts"]:
    sigs = " | ".join(str(int(s)) for s in d["signals"])
    print(f"{d['name']:<25} {d['fep']:>5}  {d['risk']:<10}  {sigs}")

print()
print("ICB baseline indicators:")
for k, v in output["icb_baseline"].items():
    if isinstance(v, dict) and v.get("value"):
        eng = f" (England: {v['england_value']})" if v.get('england_value') else ""
        print(f"  {k:<30} {v['value']}{eng}")
    elif isinstance(v, dict) and isinstance(list(v.values())[0], dict):
        print(f"  Prescribing:")
        for pk, pv in v.items():
            if pv.get("rate"):
                print(f"    {pk:<25} {pv['rate']} items/1000 ({pv.get('period','')})")


## 10. Done ✓

The file `kent-fep-data.json` is now live in `silegrand/assistivagents`.

### To wire this into the map tool:
The next step is updating `layer2-map-pdf.html` to load from this JSON file
instead of the baked-in synthetic data. That's a one-line URL change.

### To refresh quarterly:
1. Open this notebook in Colab
2. `Runtime → Run all`
3. Done — the JSON file in GitHub updates automatically

### What improves with each run:
As OpenPrescribing and Fingertips data updates monthly, each quarterly run
will capture improved signal accuracy. When sub-ICB linkage is added in Phase 2,
district-level prescribing rates will replace the ICB-anchored estimates.

---
*Assistiv Systems Limited · resiliencetools.xyz*
